In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from model.modular import Modular
from model.solution import Solution
csv_path = 'database/cisco_small2.csv'
requirement = pd.DataFrame([{
    'code': 'requirement',
    '100': 16,
    '40': 16,
    '25': 64,
    '10': 64,
}])

df = pd.read_csv(csv_path)
modules_df = df[df['type'] == 'modular']
linecards_df = df[df['type'] == 'linecard']

print(f"{len(modules_df)} module rows")
print(f"{linecards_df['code'].nunique()} unique linecards")
print(f"available modules: {modules_df['code'].unique().tolist()}")


6 module rows
27 unique linecards
available modules: ['9808', '9804', '9516', '9508', '9504', '9400']


In [2]:
results = {}
errors = {}
module_codes = modules_df['code'].unique()

for module_code in module_codes:
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

        modular = Modular(combined_data)
        solution = Solution(modular, requirement)
        result = solution.solve(heuristic="H1")

        if result is not None and not result.empty:
            results[module_code] = result
            print(f"✓ {module_code}: Found solution with {len(result)} linecards")
        else:
            errors[module_code] = "Requirement cannot be satisfied within maxmodules limit"
            print(f"✗ {module_code}: {errors[module_code]}")
    except Exception as e:
        errors[module_code] = str(e)
        print(f"✗ {module_code}: {str(e)[:100]}")

print(f"{len(results)} successful, {len(errors)} failed")


✗ 9808: this module does not contain linecards that can solve this requirement
✗ 9804: this module does not contain linecards that can solve this requirement
✓ 9516: Found solution with 2 linecards
✓ 9508: Found solution with 2 linecards
✓ 9504: Found solution with 2 linecards
✗ 9400: this module does not contain linecards that can solve this requirement
3 successful, 3 failed


In [3]:
if results:
    print("SUCCESSFUL CONFIGURATIONS")

    for module_code, solution in results.items():
        print(module_code)

        speed_columns = [col for col in solution.columns if col not in ['code', 'value']]
        speed_columns_sorted = sorted(
            speed_columns,
            key=lambda x: float(x) if x not in ['code', 'value'] else 0,
            reverse=True
        )
        display_cols = ['code'] + speed_columns_sorted + ['value']

        display_solution = solution[display_cols].copy()
        display_solution = display_solution.fillna(0)
        for col in speed_columns_sorted + ['value']:
            display_solution[col] = display_solution[col].astype(int)

        print(display_solution.to_string(index=False))

        total_value = int(solution['value'].sum())
        print(f"\n✓ Total value (throughput*ports): {total_value}")
else:
    print("No successful configurations found.")


SUCCESSFUL CONFIGURATIONS
9516
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

✓ Total value (throughput*ports): 4480
9508
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

✓ Total value (throughput*ports): 4480
9504
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

✓ Total value (throughput*ports): 4480
